# RNA GNN Data Pipeline Demo

This notebook demonstrates how to use the RNA data pipeline for GNN training.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data import RNADataset, LabelEncoder
from torch_geometric.loader import DataLoader
import torch
import matplotlib.pyplot as plt
import numpy as np

## 1. Load Datasets

In [ ]:
# Create label encoder
label_encoder = LabelEncoder()

print(f"Number of classes: {label_encoder.num_classes}")
print(f"\nClass labels:")
for i, meta_type in enumerate(label_encoder.meta_types):
    print(f"  {i:2d}: {meta_type}")

In [ ]:
# Load train dataset
train_dataset = RNADataset(
    root='../data/processed/train',
    fold_labels_path='../data/splits/train_labels.json',
    rfam_types_path='../rfam/rfam_types_full.pkl',
    st_files_dir='../data/unzipped/bpRNA_1m_90_STAFILES',
    label_encoder=label_encoder,
)

print(f"Train dataset size: {len(train_dataset)}")

In [ ]:
# Load val and test datasets
val_dataset = RNADataset(
    root='../data/processed/val',
    fold_labels_path='../data/splits/val_labels.json',
    rfam_types_path='../rfam/rfam_types_full.pkl',
    st_files_dir='../data/unzipped/bpRNA_1m_90_STAFILES',
    label_encoder=label_encoder,
)

test_dataset = RNADataset(
    root='../data/processed/test',
    fold_labels_path='../data/splits/test_labels.json',
    rfam_types_path='../rfam/rfam_types_full.pkl',
    st_files_dir='../data/unzipped/bpRNA_1m_90_STAFILES',
    label_encoder=label_encoder,
)

print(f"Val dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

## 2. Inspect a Sample

In [ ]:
# Get first sample
sample = train_dataset[0]

print("Sample structure:")
print(f"  Node features (x): {sample.x.shape}")
print(f"  Edge indices: {sample.edge_index.shape}")
print(f"  Edge attributes: {sample.edge_attr.shape}")
print(f"  Label (y): {sample.y}")
print(f"\nMetadata:")
print(f"  BPRNA ID: {sample.bprna_id}")
print(f"  RFAM ID: {sample.rfid}")
print(f"  Meta-type: {sample.meta_type}")
print(f"  Label name: {label_encoder.decode(sample.y.item())}")
print(f"  Sequence length: {sample.sequence_length}")

## 3. Analyze Dataset Statistics

In [ ]:
# Get statistics for each split
train_stats = train_dataset.get_statistics()
val_stats = val_dataset.get_statistics()
test_stats = test_dataset.get_statistics()

print("Dataset statistics:")
print(f"  Train: {train_stats['num_samples']} samples")
print(f"  Val:   {val_stats['num_samples']} samples")
print(f"  Test:  {test_stats['num_samples']} samples")

In [ ]:
# Plot class distribution
train_dist = train_stats['label_distribution']

# Sort by count
sorted_types = sorted(train_dist.items(), key=lambda x: x[1], reverse=True)
types = [t[0] for t in sorted_types]
counts = [t[1] for t in sorted_types]

plt.figure(figsize=(14, 6))
plt.bar(range(len(types)), counts)
plt.xticks(range(len(types)), types, rotation=90, ha='right')
plt.xlabel('RNA Meta-Type')
plt.ylabel('Count (log scale)')
plt.yscale('log')
plt.title('Class Distribution in Training Set')
plt.tight_layout()
plt.show()

## 4. Create DataLoaders

In [ ]:
# Create dataloaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Inspect a batch
batch = next(iter(train_loader))

print("Batch structure:")
print(f"  Total nodes: {batch.x.shape[0]}")
print(f"  Node features: {batch.x.shape}")
print(f"  Total edges: {batch.edge_index.shape[1]}")
print(f"  Batch vector: {batch.batch.shape}")
print(f"  Labels: {batch.y.shape}")
print(f"  Num graphs: {batch.num_graphs}")

## 5. Analyze Node and Edge Features

In [ ]:
# Analyze node features from first sample
sample = train_dataset[0]

# Node features are [num_nodes, 14]
# First 5 dims: nucleotide type (A, U, G, C, N)
# Next 7 dims: structure (E, S, H, I, M, B, X)
# Next 1 dim: pseudoknot indicator
# Last 1 dim: position encoding

nucleotide_features = sample.x[:, :5]
structure_features = sample.x[:, 5:12]
pseudoknot_features = sample.x[:, 12]
position_features = sample.x[:, 13]

print("Node feature breakdown:")
print(f"  Nucleotide encoding: {nucleotide_features.shape}")
print(f"  Structure encoding: {structure_features.shape}")
print(f"  Pseudoknot indicator: {pseudoknot_features.shape}")
print(f"  Position encoding: {position_features.shape}")

In [ ]:
# Analyze edge types
edge_types = sample.edge_attr.squeeze().numpy()
unique_types, counts = np.unique(edge_types, return_counts=True)

edge_type_names = {
    0: 'Backbone',
    1: 'Base pair ()',
    2: 'Pseudoknot []',
    3: 'Pseudoknot {}',
    4: 'Pseudoknot <>',
}

print("Edge type distribution:")
for edge_type, count in zip(unique_types, counts):
    print(f"  {edge_type_names.get(edge_type, f'Unknown {edge_type}')}: {count} edges")

## 6. Sequence Length Distribution

In [ ]:
# Sample sequence lengths from training set
seq_lengths = []
for i in range(min(1000, len(train_dataset))):
    sample = train_dataset[i]
    seq_lengths.append(sample.sequence_length)

plt.figure(figsize=(10, 5))
plt.hist(seq_lengths, bins=50, edgecolor='black')
plt.xlabel('Sequence Length (nucleotides)')
plt.ylabel('Count')
plt.title('Distribution of RNA Sequence Lengths (first 1000 samples)')
plt.axvline(np.mean(seq_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(seq_lengths):.1f}')
plt.axvline(np.median(seq_lengths), color='green', linestyle='--', label=f'Median: {np.median(seq_lengths):.1f}')
plt.legend()
plt.show()

print(f"Sequence length statistics (first 1000 samples):")
print(f"  Min: {min(seq_lengths)}")
print(f"  Max: {max(seq_lengths)}")
print(f"  Mean: {np.mean(seq_lengths):.2f}")
print(f"  Median: {np.median(seq_lengths):.2f}")

## 7. Ready for GNN Training!

The data pipeline is now complete. You can use these dataloaders to train your GNN model.

Example model architecture:

```python
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class RNAGNN(torch.nn.Module):
    def __init__(self, num_node_features=14, num_classes=23):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, 64)
        self.conv2 = GCNConv(64, 64)
        self.conv3 = GCNConv(64, 32)
        self.fc = torch.nn.Linear(32, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)
        
        x = global_mean_pool(x, batch)
        x = self.fc(x)
        return x
```